All the data is from: https://www.macrotrends.net/stocks/charts/WMT/walmart/revenue

In [39]:
#Walmart Annual Revenue (Millions of US $)

import pandas as pd

data = {
    "year": [2026, 2025, 2024, 2023, 2022, 2021, 2020, 2019,
              2018, 2017, 2016, 2015, 2014, 2013, 2012],
    "revenue": [713163, 680985, 648125, 611289, 572754, 559151,
                523964, 514405, 500343, 485873, 482130, 485651,
                476294, 468651, 446509]
}

df_revenue = pd.DataFrame(data)
df_revenue = df_revenue.sort_values("year").reset_index(drop=True)

df_revenue = df_revenue[df_revenue["year"]<=2023]

In [42]:
df_revenue.head(12)

,year,revenue
0,2012,446509
1,2013,468651
2,2014,476294
3,2015,485651
4,2016,482130
5,2017,485873
6,2018,500343
7,2019,514405
8,2020,523964
9,2021,559151


In [43]:
df_inflation = pd.read_csv("../../Data/inflation.csv", skiprows=4)

In [44]:
df_inflation.head()

,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2017,2018,2019,2020,2021,2022,2023,2024,2025,Unnamed: 70
0,Aruba,ABW,"Inflación, precios al consumidor (% anual)",FP.CPI.TOTL.ZG,NaN,NaN,NaN,NaN,NaN,NaN,...,-1.028282,3.626041,4.257462,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,AFE,"Inflación, precios al consumidor (% anual)",FP.CPI.TOTL.ZG,NaN,NaN,NaN,NaN,NaN,NaN,...,6.221375,4.689806,4.102851,5.191629,6.824727,10.883478,7.399186,4.770857,NaN,NaN
2,Afganistán,AFG,"Inflación, precios al consumidor (% anual)",FP.CPI.TOTL.ZG,NaN,NaN,NaN,NaN,NaN,NaN,...,4.975952,0.626149,2.302373,5.601888,5.133203,13.712102,-4.644709,-6.601186,NaN,NaN
3,NaN,AFW,"Inflación, precios al consumidor (% anual)",FP.CPI.TOTL.ZG,NaN,NaN,NaN,NaN,NaN,NaN,...,1.725486,1.784050,1.983092,2.490378,3.745568,7.949251,5.221168,3.608044,NaN,NaN
4,Angola,AGO,"Inflación, precios al consumidor (% anual)",FP.CPI.TOTL.ZG,NaN,NaN,NaN,NaN,NaN,NaN,...,29.844480,19.628938,17.080954,22.271539,25.754295,21.355290,13.644102,28.240495,NaN,NaN


In [45]:
# 2. Creamos la lista de columnas que queremos conservar
# Genera ['2010', '2011', ..., '2026'] revisando cuáles existen en tu CSV
anyos_deseados = [str(year) for year in range(2010, 2024)]
columnas_a_sacar = [col for col in anyos_deseados if col in df_inflation.columns]

# 3. Filtramos por Estados Unidos ("USA")
df_usa = df_inflation[df_inflation['Country Code'] == 'USA']

# 4. Nos quedamos SOLO con las columnas de los años que quieres
df_usa_filtrado = df_usa[columnas_a_sacar]

# 5. Lo giramos para que quede en formato vertical: Año e Inflación
df_final = df_usa_filtrado.melt(var_name='year', value_name='inflation')

# Convertimos el año a número entero para que quede limpio
df_final['year'] = df_final['year'].astype(int)



In [48]:
df_final.head(15)

,year,inflation
0,2010,1.640043
1,2011,3.156842
2,2012,2.069337
3,2013,1.464833
4,2014,1.622223
5,2015,0.118627
6,2016,1.261583
7,2017,2.130110
8,2018,2.442583
9,2019,1.812210


In [49]:
# 1. JUNTAR ELS DATAFRAMES (El Merge)
# Fem servir la columna 'year' perquè les files coincideixin exactament
df_mix = pd.merge(df_revenue, df_final, on='year')

# 2. PREPARAR EL MULTIPLICADOR
# Passem l'Inflation_Rate (ex: 2.5%) a un multiplicador matemàtic (1.025)
df_mix['Multiplicador'] = 1 + (df_mix['inflation'] / 100)

# 3. CREAR L'ÍNDEX ACUMULAT (L'efecte "bola de neu")
# Multipliquem cada any per l'anterior per veure com s'acumula la inflació amb el temps
df_mix['Index_Inflacio_Acumulada'] = df_mix['Multiplicador'].cumprod()

# 4. CALCULAR EL REVENUE REAL (Sense inflació)
# Dividim els ingressos pel cost acumulat de la vida
df_mix['Revenue_Ajustat'] = df_mix['revenue'] / df_mix['Index_Inflacio_Acumulada']

# Arrodonim perquè quedi un número sencer i net
df_mix['Revenue_Ajustat'] = df_mix['Revenue_Ajustat'].round(0).astype(int)

# Vist cop d'ull al resultat final
print(df_mix[['year', 'revenue', 'inflation', 'Revenue_Ajustat']])

    year  revenue  inflation  Revenue_Ajustat
0   2012   446509   2.069337           437457
1   2013   468651   1.464833           452521
2   2014   476294   1.622223           452559
3   2015   485651   0.118627           460903
4   2016   482130   1.261583           451861
5   2017   485873   2.130110           445872
6   2018   500343   2.442583           448203
7   2019   514405   1.812210           452597
8   2020   523964   1.233584           455390
9   2021   559151   4.697859           464166
10  2022   572754   8.002800           440228
11  2023   611289   4.116338           451271


In [ ]:
df_revenue.to_csv("../../Data/walmart_annual_revenue.csv", index=False)

In [24]:
import pandas as pd

data_quarterly = """2026-04-30,177751
2026-01-31,190656
2025-10-31,179496
2025-07-31,177402
2025-04-30,165609
2025-01-31,180554
2024-10-31,169588
2024-07-31,169335
2024-04-30,161508
2024-01-31,173388
2023-10-31,160804
2023-07-31,161632
2023-04-30,152301
2023-01-31,164048
2022-10-31,152813
2022-07-31,152859
2022-04-30,141569
2022-01-31,152871
2021-10-31,140525
2021-07-31,141048
2021-04-30,138310
2021-01-31,152079
2020-10-31,134708
2020-07-31,137742
2020-04-30,134622
2020-01-31,141671
2019-10-31,127991
2019-07-31,130377
2019-04-30,123925
2019-01-31,138793
2018-10-31,124894
2018-07-31,128028
2018-04-30,122690
2018-01-31,136267
2017-10-31,123179
2017-07-31,123355
2017-04-30,117542
2017-01-31,130936
2016-10-31,118179
2016-07-31,120854
2016-04-30,115904
2016-01-31,129667
2015-10-31,117408
2015-07-31,120229
2015-04-30,114826
2015-01-31,131565
2014-10-31,119001
2014-07-31,120125
2014-04-30,114960
2014-01-31,129706
2013-10-31,115688
2013-07-31,116830
2013-04-30,114070
2013-01-31,127559
2012-10-31,113800
2012-07-31,114282
2012-04-30,113010
2012-01-31,122728
2011-10-31,110226
2011-07-31,109366"""

from io import StringIO
df_quarterly = pd.read_csv(StringIO(data_quarterly), names=["date", "revenue"])
df_quarterly["date"] = pd.to_datetime(df_quarterly["date"])
df_quarterly = df_quarterly.sort_values("date").reset_index(drop=True)

print("Desde:", df_quarterly["date"].min())
print("Hasta:", df_quarterly["date"].max())
print(df_quarterly)

Desde: 2011-07-31 00:00:00
Hasta: 2026-04-30 00:00:00
         date  revenue
0  2011-07-31   109366
1  2011-10-31   110226
2  2012-01-31   122728
3  2012-04-30   113010
4  2012-07-31   114282
5  2012-10-31   113800
6  2013-01-31   127559
7  2013-04-30   114070
8  2013-07-31   116830
9  2013-10-31   115688
10 2014-01-31   129706
11 2014-04-30   114960
12 2014-07-31   120125
13 2014-10-31   119001
14 2015-01-31   131565
15 2015-04-30   114826
16 2015-07-31   120229
17 2015-10-31   117408
18 2016-01-31   129667
19 2016-04-30   115904
20 2016-07-31   120854
21 2016-10-31   118179
22 2017-01-31   130936
23 2017-04-30   117542
24 2017-07-31   123355
25 2017-10-31   123179
26 2018-01-31   136267
27 2018-04-30   122690
28 2018-07-31   128028
29 2018-10-31   124894
30 2019-01-31   138793
31 2019-04-30   123925
32 2019-07-31   130377
33 2019-10-31   127991
34 2020-01-31   141671
35 2020-04-30   134622
36 2020-07-31   137742
37 2020-10-31   134708
38 2021-01-31   152079
39 2021-04-30   138310
40 

In [25]:
df_quarterly.to_csv("../../Data/walmart_quarterly_revenue.csv", index=False)